# 03 · Run the active-learning mode-shape loop ON THE INSTRUMENT

Drives an Asylum/Cypher system through Igor Pro to reconstruct the contact-resonance
mode shape from the **minimum number of laser positions**, choosing each position by
active learning and stopping when the displacement null spot (D-NS) is pinned.

**Windows / Igor only.** Requires `win32com` and `igor2`, and the same environment
your dense-sweep notebook runs in.

### Before you run — set up in Igor (once)
1. Approach and find the contact resonance at your chosen load; confirm the tip is well-behaved.
2. **Set a FIXED, WIDE tune range** that brackets the whole resonance **and** the
   antiresonance across the positions you will scan (e.g. the 254-449 kHz window used
   for the dense sweep). Do **not** use resonance tracking / auto-recenter — every
   position must be tuned over the *same* frequency window.
3. Park the detection laser at the **start** of the span you want to map (positions
   below are absolute micrometers measured from here, increasing toward the tip).
4. Make sure the save folder exists and the base filename is set as usual.

## 1 · Imports and connect to Igor

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import matplotlib.pyplot as plt
import win32com.client
from activemodemap.asylum import AFMLaserSweepAutomation, AsylumInstrument
from activemodemap import LowRankModeMap, plot_state

igor = win32com.client.Dispatch('IgorPro.Application')
print('Connected to Igor Pro')

## 2 · Experiment parameters — edit these

In [ ]:
# --- file locations (same convention as the dense-sweep notebook) ---
file_loc      = r'D:\User Data\MARTI\2026\ActiveModeMap\demo'   # must exist
base_filename = 'AMap'
log_filename  = os.path.join(file_loc, 'AMap_log')

# --- contact / drive ---
load_nN   = 1000      # applied load (nN)
dc_bias_V = 0.0       # DC bias on Output.A (single-domain D-NS map)
tune_center_Hz = 350000    # for logging only; the actual window is whatever you set in Igor

# --- accessible laser-position grid (ABSOLUTE um from the start position) ---
span_um   = 30.0      # total span to map, starting at the current laser position
step_um   = 1.0       # candidate grid spacing (the loop visits a subset of these)
x_grid    = np.arange(0.0, span_um + 1e-6, step_um)

# --- active-learning settings ---
RANK          = 5       # spatial basis dimension
MAX_POSITIONS = 10      # hard cap on positions actually measured
MIN_POSITIONS = 6       # do not stop before this many
DNS_CI_TOL_UM = 1.0     # stop once the D-NS 95% CI is below this
RECALIBRATE_EACH = True # AutoWedge + InvOLS at every position (safer; set False to reuse first)

os.makedirs(file_loc, exist_ok=True)

## 3 · Build the automation, the instrument adapter, and the loop manager

In [ ]:
automation = AFMLaserSweepAutomation(igor, file_loc, base_filename, log_filename)
automation.eigenmode_center_freq = tune_center_Hz
automation.autowedge_pause = 15.0
automation.invols_bounds = (4e-8, 10e-7)

inst = AsylumInstrument(automation, load_nN=load_nN, dc_bias_V=dc_bias_V,
                        x_start_um=0.0, recalibrate_each=RECALIBRATE_EACH)

mm = LowRankModeMap(x_grid, freq_grid=None, rank=RANK,
                    seeds_um=[x_grid[0], x_grid[len(x_grid)//2], x_grid[-1]],
                    min_positions=MIN_POSITIONS, dns_ci_tol_um=DNS_CI_TOL_UM)
print('Ready. Grid positions:', x_grid)

## 4 · Run the active-learning loop

Each iteration: pick the next position, drive the instrument (withdraw → move laser →
optical image → AutoWedge+InvOLS → engage → tune → read the complex spectrum →
withdraw), update the reconstruction, and decide whether to continue. Interrupt the
cell at any time; whatever has been measured is retained in `mm` and `inst.records`.

In [ ]:
rec = None
for step in range(MAX_POSITIONS):
    x = mm.next_position()
    print(f'\n=== position {mm.n+1}/{MAX_POSITIONS}:  x = {x:.1f} um ===')
    freq, Z, meta = inst.measure_at(x)
    mm.add_measurement(x, freq, Z)
    if meta.get('resonance_freq_Hz'):
        print(f"    f_res = {meta['resonance_freq_Hz']/1e3:.2f} kHz, "
              f"InvOLS = {meta['invols_m_per_V']:.2e} m/V")
    if mm.n >= mm.min_positions:
        rec = mm.reconstruct(nboot=150)
        print(f'    D-NS estimate: {rec["dns"]:.2f} um  (95% CI {rec["dns_ci"]:.2f} um)')
        try:
            from IPython.display import clear_output; clear_output(wait=True)
        except Exception: pass
        ax = plot_state(mm, rec, title=f'{mm.n} positions'); plt.tight_layout(); plt.show()
        if mm.converged():
            print('\nConverged — D-NS confidence interval below tolerance.'); break
inst.close()   # withdraw + zero DC bias
print('\nLoop done. Positions measured:', mm.n)

## 5 · Final reconstruction and save

In [ ]:
rec = mm.reconstruct(nboot=400)
x, f = mm.x_grid, mm.freq/1e3
ext = [x[0], x[-1], f[0], f[-1]]
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
amp = np.abs(rec['Zrec']).T
ax[0].imshow(np.log10(amp+1e-12), origin='lower', aspect='auto', extent=ext, cmap='viridis')
[ax[0].axvline(xs, color='w', lw=0.8) for xs in rec['x_sel']]
ax[0].axvline(rec['dns'], color='#e34948', ls='--')
ax[0].set_title(f'reconstructed mode shape ({mm.n} positions)')
ax[1].imshow((rec['std']/(amp.max()+1e-12)).T, origin='lower', aspect='auto', extent=ext, cmap='magma')
ax[1].set_title('uncertainty (1sigma, rel.)')
ax[2].hist(rec['dns_samples'], bins=16, color='#2a78d6', alpha=0.75, density=True)
ax[2].axvline(rec['dns'], color='k')
ax[2].set_title(f"D-NS = {rec['dns']:.2f} +/- {rec['dns_ci']/2:.2f} um")
[a.set_xlabel('position (um)') for a in ax[:2]]; ax[0].set_ylabel('frequency (kHz)')
plt.tight_layout()
fig.savefig(os.path.join(file_loc, 'ActiveModeMap_result.png'), dpi=150, bbox_inches='tight')

# save the reconstruction, the raw measured spectra, and the per-position log
sel = rec['sel_idx']
np.savez(os.path.join(file_loc, 'ActiveModeMap_result.npz'),
         x_grid=mm.x_grid, freq=mm.freq, Zrec=rec['Zrec'], std=rec['std'],
         measured_x=mm.x_grid[sel],
         measured_Z=np.array([mm._measured[i] for i in sel]),
         dns=rec['dns'], dns_ci=rec['dns_ci'])
log_path = inst.save_log()
print('saved:  ActiveModeMap_result.png / .npz  and', os.path.basename(log_path))
print(f'D-NS = {rec["dns"]:.2f} +/- {rec["dns_ci"]/2:.2f} um  from {mm.n} positions')

## Notes and extensions

- **D-ESBS (electrostatic blind spot).** This notebook maps the D-NS from a single-
  domain tune series. To also locate the D-ESBS, measure both PPLN domain orientations
  at each position (as in the dense-sweep protocol) and separate the piezoresponse and
  electrostatic channels via their difference and sum; the same low-rank reconstruction
  then applies to each channel.
- **Load / bias series.** Wrap this loop in an outer loop over loads or biases, re-
  seeding `mm` each time (the D-NS migrates with load).
- **Physics-informed option.** For a well-characterized probe you can instead drive the
  loop with the physics-informed reconstruction (`activemodemap.loop`), which needs
  fewer positions and returns contact stiffness and the drive ratio, at the cost of
  sensitivity to model error. See the paper's Methods and the simulation notebooks.